# Padronização, Preparação e Validação da Landing Zone

In [0]:
#padronizando os nomes como feito na aula prática
catalog = "movies_medallion"
bronze_schema_name = "bronze"
silver_schema_name = "silver"
gold_schema_name = "gold"

bronze_schema = f"{catalog}.{bronze_schema_name}"
silver_schema = f"{catalog}.{silver_schema_name}"
gold_schema = f"{catalog}.{gold_schema_name}"

landing_path = f"/Volumes/{catalog}/{bronze_schema_name}/Landing"

print(f"catalog: {catalog}")
print(f"bronze_schema: {bronze_schema}")
print(f"silver_schema: {silver_schema}")
print(f"gold_schema: {gold_schema}")
print(f"landing_path: {landing_path}")

catalog: movies_medallion
bronze_schema: movies_medallion.bronze
silver_schema: movies_medallion.silver
gold_schema: movies_medallion.gold
landing_path: /Volumes/movies_medallion/bronze/Landing


## Criação do Catalog, Schemas e Volume

In [0]:
spark.sql(f"CREATE CATALOG IF NOT EXISTS {catalog}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {bronze_schema}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {silver_schema}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {gold_schema}")
spark.sql(f"CREATE VOLUME IF NOT EXISTS {bronze_schema}.Landing")

DataFrame[]

## Validação da Landing Zone

In [0]:
import re

#Utiliza o módulo re (regex) para corrigir possíveis títulos errados antes de verificar a presença do arquivo.
#Cria o dicionário contendo os nomes corretos e os regex para encontrar os nomes errados.
expected_files = {
    "movies_info_IMDB_TMDB.csv":        r"movies_info_.*\.csv$",
    "movies_financials_IMDB_TMDB.csv":  r"movies_financials_.*\.csv$",
    "movies_metrics_IMDB_TMDB.csv":     r"movies_metrics_.*\.csv$",
    "credits_and_tags_IMDB_TMDB.csv":   r"credits_and_tags_.*\.csv$",
    "movies_reviews.csv":               r"movies_reviews.*\.csv$",
}

#Verifica se todos os arquivos estão presentes e corrige o nome se necessário.
found_files = [f.name for f in dbutils.fs.ls(landing_path)]
print("Encontrados:", found_files)

missing_files = []

for right_name, regex in expected_files.items():
    if right_name in found_files:
        continue

    wrong_name = None
    for i in found_files:
        if re.match(regex, i):
            wrong_name = i
            break

    if wrong_name:
        print(f"Renomeando {wrong_name} -> {right_name}")
        dbutils.fs.mv(f"{landing_path}/{wrong_name}", f"{landing_path}/{right_name}")
        found_files.remove(wrong_name)
        found_files.append(right_name)
    else:
        missing_files.append(right_name)

if missing_files:
    raise Exception(f"Os seguintes arquivos estão faltando: {missing_files}")
else:
    print("Todos os arquivos estão presentes.")
 

Encontrados: ['cotacao_dolar.json', 'credits_and_tags_IMDB_TMDB.csv', 'movies_financials_IMDB_TMDB.csv', 'movies_info_IMDB_TMDB.csv', 'movies_metrics_IMDB_TMDB.csv', 'movies_reviews.csv']
Todos os arquivos estão presentes.
